# Lichess Skill-Model Data Pipeline

Builds the modelling table for **Beyond Glicko-2: Hierarchical Bayesian
Estimation of Latent Chess Skill**.

Two arms, one pipeline:

| Arm | Question | Columns it needs |
|---|---|---|
| **A — information-fair audit** | Does proper joint inference beat Glicko-2 on the *same* data? | outcome, players, timestamp, time class, **pre-game Glicko rating** |
| **B — value of information** | What is Glicko-2 leaving on the table? | termination type, ply count, per-side clock usage |

**Four stages.** Stages 1 and 3 are the expensive ones; you run each once.

```
1. discovery   stream corpus -> per-player activity counts     (~1-3 h, once)
2. cohort      DuckDB -> connected player list                 (seconds)
3. enrich      re-stream -> cohort games + movetext features   (~2-4 h, once)
4. assemble    -> model-ready .npz                             (~1 min)
```

**Nothing large ever touches disk.** The `.zst` is streamed over HTTP and
decompressed in memory. A 30 GB compressed / 200 GB decompressed month-file is
processed with a constant ~50 MB footprint, so Colab's ephemeral disk is never
a constraint.

> **Run stages 1 and 3 on a machine you control if you can.** They are pure
> I/O — no GPU, no big RAM — and Colab session death across a 3-hour download
> loop buys you nothing. The manifest makes restarts cheap, but a laptop or a
> cheap spot VM is the better host. Stage 4 and all modelling belong on Colab.

## 0 · Setup

In [ ]:
!pip -q install zstandard duckdb pyarrow requests

import os, sys, json, time
from google.colab import drive
drive.mount('/content/drive')

PKG  = '/content/drive/MyDrive/Colab Notebooks/Bayes ML Final Project'
ROOT = '/content/drive/MyDrive/Colab Notebooks/Bayes ML Final Project/data'
os.makedirs(ROOT, exist_ok=True)
sys.path.insert(0, PKG)

# Shell-safe versions for the ! cells (handles the spaces in the folder name)
import shlex
PKG_Q, ROOT_Q = shlex.quote(PKG), shlex.quote(ROOT)

import lichess_stream, importlib
importlib.reload(lichess_stream)
print('ok —', len(os.listdir(PKG)), 'files visible')

Mounted at /content/drive
ok — 9 files visible


## 1 · Smoke test — Reads ~200 MB of one month (a few hundred thousand games) and prints parsed records. Takes about a minute. **Never launch the overnight run without seeing this output look right. **Check specifically:- `TimeControl` parses into sensible classes- `Termination` includes `Time forfeit` — this is Arm B's whole beta/kappa split- `[%clk]` tags are present in the movetext of the month you plan to use

In [ ]:
from lichess_stream import open_month, iter_games, time_class, clock_features

MONTH = '2026-05'   # <- pick a month inside your window

for i, (h, mv) in enumerate(iter_games(open_month(MONTH, max_bytes=20<<20),
                                       want_movetext=True)):
    print(h)
    print('  class      :', time_class(h.get('TimeControl')))
    print('  termination:', h.get('Termination'))
    if mv:
        print('  clocks     :', clock_features(mv, h.get('TimeControl')))
        print('  movetext[:120]:', mv[:120])
    print()
    if i >= 4: break

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/NazNQks5', 'Site': 'https://lichess.org/nBqgwfXu', 'White': 'poptartkilla', 'Black': 'OxRook', 'Result': '0-1', 'UTCDate': '2026.05.01', 'UTCTime': '00:00:22', 'WhiteElo': '2219', 'BlackElo': '2220', 'WhiteRatingDiff': '-6', 'BlackRatingDiff': '+6', 'ECO': 'D00', 'TimeControl': '30+0', 'Termination': 'Time forfeit'}
  class      : bullet
  termination: Time forfeit
  clocks     : {'n_plies': 78, 'has_clock': True, 'w_clk_final': 1, 'w_clk_min': 1, 'w_move_t_mean': 0.7631578947368421, 'w_move_t_max': 2, 'w_frac_under_10s': 0.23076923076923078, 'b_clk_final': 1, 'b_clk_min': 1, 'b_move_t_mean': 0.7631578947368421, 'b_move_t_max': 2, 'b_frac_under_10s': 0.15384615384615385, 'base_seconds': 30}
  movetext[:120]: b'1. d4 { [%clk 0:00:30] } 1... d5 { [%clk 0:00:30] } 2. e3 { [%clk 0:00:30] } 2... Nf6 { [%clk 0:00:30] } 3. Bd3 { [%clk '

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/NazNQks5', 'Site': 'https

In [ ]:
# Class + termination distribution on a small sample - sanity check the buckets.
# NOTE: this reads the HEAD of the month file, which is time-ordered, so it is
# a snapshot of the first minutes of the month (arena-heavy), not a
# representative sample. Use it to confirm the buckets work, not to estimate
# proportions.
from collections import Counter

cls, term, clk, arena = Counter(), Counter(), Counter(), Counter()

for n, (h, mv) in enumerate(iter_games(open_month(MONTH, max_bytes=200 << 20),
                                       want_movetext=True)):
    cls[time_class(h.get('TimeControl'))] += 1
    term[h.get('Termination')] += 1
    clk[bool(mv and b'%clk' in mv)] += 1
    arena['tournament' in h.get('Event', '')] += 1

print('time class :', cls.most_common())
print('termination:', term.most_common())
print('has [%clk] :', dict(clk))
print('arena game :', dict(arena))
print('games      :', n + 1)

time class : [('blitz', 295365), ('bullet', 228724), ('rapid', 109915), ('ultrabullet', 6651), ('classical', 4444), ('correspondence', 615)]
termination: [('Normal', 431892), ('Time forfeit', 211877), ('Abandoned', 1764), ('Insufficient material', 111), ('Unterminated', 52), ('Rules infraction', 18)]
has [%clk] : {True: 643900, False: 1814}
arena game : {True: 57510, False: 588204}
games      : 645714


## 2 · Discovery passStreams every month and writes **only** per-player activity aggregates — no game rows.Output is ~60-100 MB per month, so the whole window fits in Drive.Resumable: completed months are recorded in `_manifest.json` and skipped on re-run.If the session dies, just re-execute the cell.

In [ ]:
WINDOW_START, WINDOW_END = '2025-06', '2026-05'   # 12 months
COHORT_CLASS = 'blitz'                            # highest volume -> densest graph

# !python {PKG_Q}/pass1_discovery.py \
#     --start {WINDOW_START} --end {WINDOW_END} \
#     --time-class {COHORT_CLASS} \
#     --out {ROOT_Q}/discovery

## 3 · Cohort selection: **The single most important design decision in the project.** You are not maximising player count — you are getting a *connected* comparison graph of workable size. Relative skill is identified only through chains of shared opponents. Targets: **5k–20k players**, and after stage 4, **10⁵–10⁶ games**. Tune `--min-games` until you land in that range, then stop.

In [ ]:
!python {PKG_Q}/cohort.py \
    --discovery {ROOT_Q}/discovery \
    --out {ROOT_Q}/cohort.parquet \
    --min-games 5000 \
    --min-active-months 6

distinct players in window: 3,891,003
cohort: 21,603 players | 156,027,711 player-games
  mean-elo range: 625 / 1781 (median) / 3025

rating strata:
    600-799         26
    800-999        211
   1000-1199       749
   1200-1399     1,827
   1400-1599     3,387
   1600-1799     5,168
   1800-1999     5,666
   2000-2199     3,251
   2200-2399     1,036
   2400-2599       231
   2600-2799        23
   2800-2999        20
   3000-3199         8

wrote /content/drive/MyDrive/Colab Notebooks/Bayes ML Final Project/data/cohort.parquet


In [ ]:
import duckdb
con = duckdb.connect()
n = con.execute(f"""
    SELECT COUNT(*), AVG(n_games), MIN(n_games), AVG(active_months)
    FROM read_parquet('{ROOT}/cohort.parquet')
""").fetchone()
print(f"players {n[0]:,} | avg games {n[1]:.0f} | min {n[2]:.0f} | avg active months {n[3]:.1f}")

players 21,603 | avg games 7223 | min 5000 | avg active months 11.6


## 4 · Enrichment passRe-streams the corpus. Keeps a game only if **both** players are in the cohort — roughly0.1–1% of games — and captures all four time classes, since the cross-format covarianceΣ is a headline result.`--with-movetext` is what buys Arm B. Because the header is read before the decision tocapture, movetext is parsed only for cohort games; the other 99%+ have their move linesrecognised and dropped without being copied. That is why enrichment costs ~2-3× discoveryrather than ~50×.Drop the flag to build Arm A only.

In [ ]:
!python {PKG_Q}/pass2_enrich.py \
    --start {WINDOW_START} --end {WINDOW_END} \
    --cohort {ROOT_Q}/cohort.parquet \
    --out {ROOT_Q}/games \
    --with-movetext

cohort loaded: 21,603 players
[2025-06] already done, skipping
[2025-07] already done, skipping
[2025-08] already done, skipping
[2025-09] streaming...
  [2025-09] 5M scanned, 77,405 kept, 29k games/s
  [2025-09] 10M scanned, 155,043 kept, 29k games/s
  [2025-09] 15M scanned, 229,373 kept, 29k games/s
  [2025-09] 20M scanned, 310,718 kept, 29k games/s
  [2025-09] 25M scanned, 393,654 kept, 29k games/s
  [2025-09] 30M scanned, 475,253 kept, 29k games/s
  [2025-09] 35M scanned, 554,351 kept, 29k games/s
  [2025-09] 40M scanned, 635,183 kept, 29k games/s
  [2025-09] 45M scanned, 719,465 kept, 29k games/s
  [2025-09] 50M scanned, 800,419 kept, 29k games/s
  [2025-09] 55M scanned, 878,936 kept, 29k games/s
  [2025-09] 60M scanned, 962,548 kept, 29k games/s
  [2025-09] 65M scanned, 1,046,349 kept, 29k games/s
  [2025-09] 70M scanned, 1,128,771 kept, 29k games/s
  [2025-09] 75M scanned, 1,207,385 kept, 29k games/s
  [2025-09] 80M scanned, 1,290,439 kept, 29k games/s
  [2025-09] 85M scanned, 1

## 5 · Assemble Integer arrays, connectivity filter, stratum flags, and the leakage-safe split. **Read the split logic.** The FFBS sampler is a *smoother* — its posterior at time *t* conditions on games after *t*. Glicko-2 is a *filter* and only ever saw the past. Scoring smoothed estimates against Glicko-2 uses the future to predict the past and produces a fake win. This is the most likely way the project quietly produces a wrong answer thatlooks great.The terminal holdout emitted here is the honest comparison; one-step-ahead evaluationinside the training window uses the filtered state and is handled in the sampler.

In [ ]:
!python {PKG_Q}/assemble.py \
    --games {ROOT_Q}/games \
    --out {ROOT_Q}/model_data.npz \
    --holdout-months 2 \
    --bucket month \
    --subsample-frac 0.20 \
    --return-pct 99 \
    --return-min-days 3

games loaded: 16,412,774
players: 21,603 | time buckets: 12

gap distribution (days since a player's previous game):
  p50        0.09
  p90        0.98
  p95        1.57
  p99        3.60
  p99.9     11.43
  >  1d     1,544,041 games ( 9.41%)
  >  3d       228,484 games ( 1.39%)
  >  7d        39,825 games ( 0.24%)
  > 14d        11,455 games ( 0.07%)
  > 30d         3,176 games ( 0.02%)
  > 60d           973 games ( 0.01%)

returning: p99.0 per-player threshold, floor 3.0d -> median effective threshold 3.0d over 21,603 players

subsample: protecting 339,975 non-blitz training games in full
subsample: keeping 20% of the subsampled pool (2,716,458 of 13,582,288); holdout kept in full
split: train buckets 0..9 (3,056,433 games) | holdout 10..11 (2,490,511 games)
connectivity: 1 components, giant = 21,603 players (100.0%)
after connectivity filter: 5,546,944 games, 21,603 players

  stratum early_window       99,872 games ( 1.80%)  | in holdout: 0
  stratum returning          73,219 game

## 6 · Validation — check these before modelling

In [ ]:
import numpy as np

d = np.load(f'{ROOT}/model_data.npz', allow_pickle=True)

n_g, n_p, n_t = len(d['y']), int(d['n_players']), int(d['n_buckets'])
print(f'games {n_g:,} | players {n_p:,} | buckets {n_t}')
print(f'games/player {2*n_g/n_p:.0f}   <- want >= 100')

# Outcome balance. Draw rate should climb with rating and with slower time controls.
import collections
print('outcomes (0=black,1=draw,2=white):', collections.Counter(d['y'].tolist()))
print('white win rate:', (d['y'] == 2).mean().round(4),
      '| draw rate:', (d['y'] == 1).mean().round(4))

# Degree distribution: isolated-ish players contribute little and slow mixing.
deg = np.bincount(np.concatenate([d['white'], d['black']]), minlength=n_p)
print(f'degree  min {deg.min()}  p05 {np.percentile(deg, 5):.0f}  '
      f'median {np.median(deg):.0f}  max {deg.max()}')

# Coverage of the time grid: players with only one active bucket give no dynamics.
active = np.zeros((n_p, n_t), bool)
active[d['white'], d['t_idx']] = True
active[d['black'], d['t_idx']] = True
print('buckets active per player: median', np.median(active.sum(1)))

games 5,546,944 | players 21,603 | buckets 12
games/player 514   <- want >= 100
outcomes (0=black,1=draw,2=white): Counter({2: 2700993, 0: 2506462, 1: 339489})
white win rate: 0.4869 | draw rate: 0.0612
degree  min 15  p05 133  median 436  max 16279
buckets active per player: median 12.0


In [ ]:
# Arm B: is the flag-fall channel actually informative?
# If flagging rate barely varies across the rating ladder, the beta/kappa split
# has nothing to find and you should say so rather than model it anyway.
import numpy as np

elo = (d['glicko_white'].astype(float) + d['glicko_black'].astype(float)) / 2
bins = np.arange(800, 2800, 200)
idx = np.digitize(elo, bins)

print(' rating   n_games   flag-rate   draw-rate   median plies')
for k in range(1, len(bins) + 1):
    m = idx == k
    if m.sum() < 500:
        continue
    pl = d['n_plies'][m]
    pl = pl[pl > 0]
    lo = bins[k - 1]
    hi = f'{bins[k]}' if k < len(bins) else '+'
    print(f' {lo:>5}   {m.sum():>7,}   '
          f'{d["flagged"][m].mean():>9.3f}   {(d["y"][m] == 1).mean():>9.3f}   '
          f'{np.median(pl) if len(pl) else float("nan"):>11.0f}')

 rating   n_games   flag-rate   draw-rate   median plies
   800    11,322       0.340       0.033            57
  1000    60,811       0.294       0.038            62
  1200   239,037       0.270       0.043            65
  1400   613,016       0.252       0.046            67
  1600   1,245,411       0.265       0.041            69
  1800   1,707,528       0.281       0.045            72
  2000   1,111,875       0.277       0.055            75
  2200   340,281       0.252       0.076            80
  2400    94,286       0.183       0.096            92
  2600   122,245       0.043       0.600           108


## 7 · The Glicko-2 baseline tableExtracted here so the baseline ladder is built from the same rows the model sees —the pre-game ratings in the PGN headers are what Glicko-2 believed *before* the outcome,so scoring them is genuinely prospective. No refitting, no leakage.Fit the draw margin γ and white advantage *h* on the training split only. Without themyou would be reporting a win that is really just "Glicko-2 has no draw model", which isa known modelling gap rather than an approximation failure — and a professor will say so.

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

tr = d['is_train']

# Glicko-2 rating difference on the probit scale (400-point logistic -> ~ /173.7)
diff = (d['glicko_white'].astype(float) - d['glicko_black'].astype(float)) / 173.7

def nll(params, m):
    h, log_g, s = params
    g = np.exp(log_g)
    z = s * diff[m] + h
    pW = norm.cdf(z - g)
    pL = norm.cdf(-z - g)
    p = np.stack([pL, np.clip(1 - pW - pL, 1e-9, 1), pW])
    return -np.log(p[d['y'][m], np.arange(m.sum())] + 1e-12).mean()

fit = minimize(nll, [0.05, np.log(0.3), 1.0], args=(tr,), method='Nelder-Mead')
h_hat, gamma_hat, scale_hat = fit.x[0], np.exp(fit.x[1]), fit.x[2]

print(f'B2 baseline: h={h_hat:.4f}  gamma={gamma_hat:.4f}  scale={scale_hat:.4f}')
print(f'train NLL {fit.fun:.4f} | holdout NLL {nll(fit.x, ~tr):.4f}')

np.savez(f'{ROOT}/glicko_baseline.npz', h=h_hat, gamma=gamma_hat, scale=scale_hat)

B2 baseline: h=0.0471  gamma=0.0913  scale=0.5276
train NLL 0.8591 | holdout NLL 0.8425


In [ ]:
# Ranked Probability Score - the right scoring rule for ORDERED outcomes.
# Report this alongside log loss; plain log loss ignores that a draw is
# "closer" to a win than a loss is.
def rps(p, y):
    cp = np.cumsum(p, axis=1)[:, :-1]
    oh = np.cumsum(np.eye(3)[y], axis=1)[:, :-1]
    return ((cp - oh) ** 2).sum(1).mean()

def glicko_probs(m, h, g, s):
    z = s * diff[m] + h
    pW = norm.cdf(z - g)
    pL = norm.cdf(-z - g)
    return np.stack([pL, np.clip(1 - pW - pL, 1e-9, 1), pW], axis=1)

for name, params in [('B0 raw    ', (0.0, 0.0, 1.0)),
                     ('B1 +gamma ', (0.0, gamma_hat, scale_hat)),
                     ('B2 +h     ', (h_hat, gamma_hat, scale_hat))]:
    p = glicko_probs(~tr, *params)
    print(f'{name}  holdout RPS {rps(p, d["y"][~tr]):.5f}')

print('\nThese are the numbers your model has to beat. Save them.')

B0 raw      holdout RPS 0.49088
B1 +gamma   holdout RPS 0.48048
B2 +h       holdout RPS 0.47988

These are the numbers your model has to beat. Save them.


In [ ]:
base = np.array([(d['y'][tr] == k).mean() for k in range(3)])
p_null = np.tile(base, (int((~tr).sum()), 1))
print(f'B-null (marginal rates)  holdout RPS {rps(p_null, d["y"][~tr]):.5f}')

B-null (marginal rates)  holdout RPS 0.49820


In [ ]:
def nll_p(h, g, s, m):
    z = s * diff[m] + h
    pW = norm.cdf(z - g); pL = norm.cdf(-z - g)
    p = np.stack([pL, np.clip(1 - pW - pL, 1e-9, 1), pW])
    return -np.log(p[d['y'][m], np.arange(m.sum())] + 1e-12).mean()

r = minimize(lambda x: nll_p(0.0, 0.0, x[0], tr), [1.0], method='Nelder-Mead')
s_b0b = r.x[0]

r = minimize(lambda x: nll_p(0.0, np.exp(x[0]), x[1], tr),
             [np.log(0.1), 0.6], method='Nelder-Mead')
g_b1, s_b1 = np.exp(r.x[0]), r.x[1]

rungs = [('B0  raw       ', (0.0,   0.0,   1.0)),
         ('B0b +scale    ', (0.0,   0.0,   s_b0b)),
         ('B1  +draw marg', (0.0,   g_b1,  s_b1)),
         ('B2  +white adv', (h_hat, gamma_hat, scale_hat))]

prev = None
for name, prm in rungs:
    v = rps(glicko_probs(~tr, *prm), d['y'][~tr])
    delta = '' if prev is None else f'  (Δ {prev - v:+.5f})'
    print(f'{name}  holdout RPS {v:.5f}{delta}')
    prev = v

B0  raw         holdout RPS 0.49088
B0b +scale      holdout RPS 0.48198  (Δ +0.00890)
B1  +draw marg  holdout RPS 0.48047  (Δ +0.00151)
B2  +white adv  holdout RPS 0.47988  (Δ +0.00059)


In [ ]:
import json

ladder = {
    'null_marginal': 0.49820,
    'B0_raw':        0.49088,
    'B0b_scale':     0.48198,
    'B1_draw':       0.48047,
    'B2_white':      0.47988,
}
params = {
    'h': float(h_hat), 'gamma': float(gamma_hat), 'scale': float(scale_hat),
    's_b0b': float(s_b0b), 'g_b1': float(g_b1), 's_b1': float(s_b1),
    'train_nll': 0.8591, 'holdout_nll': 0.8425,
}
meta = {
    'n_holdout': int((~tr).sum()),
    'n_train':   int(tr.sum()),
    'note': 'fitted on training split; bots NOT yet excluded',
}
json.dump({'ladder_rps': ladder, 'params': params, 'meta': meta},
          open(f'{ROOT}/baseline_ladder.json', 'w'), indent=2)
print(open(f'{ROOT}/baseline_ladder.json').read())

{
  "ladder_rps": {
    "null_marginal": 0.4982,
    "B0_raw": 0.49088,
    "B0b_scale": 0.48198,
    "B1_draw": 0.48047,
    "B2_white": 0.47988
  },
  "params": {
    "h": 0.0470689157094545,
    "gamma": 0.09128896960600533,
    "scale": 0.5276008807628209,
    "s_b0b": 0.5723632812499997,
    "g_b1": 0.09118785143920918,
    "s_b1": 0.5267915197547701,
    "train_nll": 0.8591,
    "holdout_nll": 0.8425
  },
  "meta": {
    "n_holdout": 2490511,
    "n_train": 3056433,
    "note": "fitted on training split; bots NOT yet excluded"
  }
}


---## What you have now`model_data.npz` — everything the sampler touches, typically 20–80 MB. From here on nothingreads Parquet in a hot loop, and the modelling fits on a T4 with room to spare (full modelstate is under 10 MB; the bottleneck was always extraction, never compute).## Before you scale upRun the whole thing end-to-end with `--smoke-test`, a 3-month window, and `--min-games 40`.You want a toy cohort of ~500 players working in a day.Then, **before pointing the sampler at real games**, simulate: generate synthetic outcomesfrom known θ trajectories and confirm the sampler recovers them with correct coverage. Themost common way this project fails is a subtly wrong sampler that runs cleanly and returnsconfidently wrong posteriors — and simulation-based calibration is the only thing thatcatches it.